In [3]:
from pathlib import Path
from eeg_music.data import EEGMusicDataset, MappedDataset, MelParams, TrialData, WavRAW, prepare_trial, wavraw_to_melspectrogram


# ds = EEGMusicDataset.load_ondisk(
#     Path("./datasets/bcmi_preprocessed/bcmi_pre_60ch/")
#   )
ds = EEGMusicDataset.load_ondisk(
    Path("./datasets/musing_preprocessed/musing_wav_60ch/")
  )

def applying_mel(trial, apply_mel: MelParams, wav_resample=None):
  music = trial.music_data.get_music()
  match music:
    case WavRAW(raw_data=raw, sample_rate=sr) as wav:
      if wav_resample is not None and wav_resample != sr:
        wav = wav.resampled(new_sr=wav_resample)
        # raw, sr = wav.raw_data, wav.sample_rate
      music_cropped = wavraw_to_melspectrogram(wav, **apply_mel.as_kwargs())
      return TrialData(
        dataset=trial.dataset,
        subject=trial.subject,
        session=trial.session,
        run=trial.run,
        trial_id=trial.trial_id,
        music_filename=trial.music_filename,
        eeg_data=trial.eeg_data,
        music_data=music_cropped,
      )
    case _:
      raise ValueError("Unexpected music data format")

mds = MappedDataset(ds, lambda t: applying_mel(
    t,
    wav_resample=32768,
    apply_mel=MelParams(n_mels=64, fmax=8192.0),
  ))


In [4]:
mds[0].eeg_data.get_array().data.shape, mds[0].music_data.mel.shape

((60, 1351), (64, 29007))

In [5]:
# mds.save(Path("./datasets/bcmi_preprocessed/bcmi_mel64_60ch/"))
mds.save(Path("./datasets/bcmi_preprocessed/musing_mel64_60ch/"))

In [27]:
from fractions import Fraction
from eeg_music.data import ArrayStratifiedSamplingDataset


strat = ArrayStratifiedSamplingDataset(mds, 10, trial_length_secs=Fraction(2, 1))
strat[0].eeg_data.get_array().data.shape, strat[0].music_data.mel.shape

((60, 20), (64, 128))

In [30]:
strat[0].music_data.mel[:, 32:64+32].shape

(64, 64)